# Demo 04 : Two agents over A2A

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · two-agents-over-a2a

We start a coordinator and a parts specialist on separate ports with separate local storage. The coordinator node sends plain httpx JSON-RPC to the specialist. The notebook does not relay the specialist's answer.

In [3]:
import atexit
import re
import subprocess
from uuid import UUID, uuid4, uuid5
import httpx

servers = []


def stop_servers():
    for process, log, handle in reversed(servers):
        if process.poll() is None:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], capture_output=True, check=False)
            else:
                os.killpg(process.pid, 15)
            process.wait(timeout=20)
        handle.close()
    servers.clear()


atexit.register(stop_servers)


NAMESPACE_GRAPH = UUID("6ba7b821-9dad-11d1-80b4-00c04fd430c8")


def start_server(port, graphs, folder, keyless=False):
    if port not in {2124, 2125}:
        raise ValueError("Use the module's reserved ports.")
    folder.mkdir(parents=True, exist_ok=True)
    config = folder / "langgraph.json"
    config.write_text(json.dumps({"dependencies": ["."], "graphs": graphs}), encoding="utf-8")
    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    if keyless:
        for key in list(env):
            if key.startswith(("LANGSMITH_", "LANGCHAIN_")):
                env.pop(key, None)
        env["LANGSMITH_TRACING"] = "false"
    log = folder / "server.tmp"
    handle = log.open("w", encoding="utf-8")
    process = subprocess.Popen(["uv", "run", "langgraph", "dev", "--port", str(port),
                                "--config", str(config), "--no-browser", "--no-reload",
                                "--allow-blocking"],
                               cwd=folder, env=env, stdout=handle, stderr=subprocess.STDOUT,
                               creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
                               start_new_session=os.name != "nt")
    servers.append((process, log, handle))
    url = f"http://127.0.0.1:{port}"
    deadline = time.monotonic() + 90
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError("Agent Server exited. See " + str(log))
        try:
            response = httpx.get(url + "/ok", timeout=3, trust_env=False)
            if response.status_code == 200:
                print("server :", url, "ready")
                return url
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    stop_servers()
    raise TimeoutError("Agent Server readiness timed out.")


def server_facts(index=0):
    text = servers[index][1].read_text(encoding="utf-8", errors="replace")
    studio = ""
    for line in text.splitlines():
        if "api_key" in line.lower() or "secret" in line.lower():
            continue
        match = re.search(r"https://[a-z.]*smith\.langchain\.com/studio/", line)
        if match:
            studio = match.group(0)
            break
    print("studio url :", studio)
    print("metadata loop skipped :", "skipping metadata loop" in text)
    print("EU metadata endpoint :", "eu.api.smith.langchain.com" in text)


def assistant_for(url, graph_id):
    response = httpx.post(url + "/assistants/search", json={"limit": 50}, timeout=30, trust_env=False)
    response.raise_for_status()
    matches = [row["assistant_id"] for row in response.json() if row.get("graph_id") == graph_id]
    if len(matches) != 1:
        raise RuntimeError("Expected one assistant for " + graph_id)
    derived = str(uuid5(NAMESPACE_GRAPH, graph_id))
    print("graph id :", graph_id)
    print("assistant id :", matches[0])
    print("uuid5 match :", matches[0] == derived)
    return matches[0]


def rpc(url, assistant, method, params):
    request = {"jsonrpc": "2.0", "id": str(uuid4()), "method": method, "params": params}
    response = httpx.post(url + "/a2a/" + assistant, json=request, timeout=180, trust_env=False,
                          headers={"Accept": "application/json"})
    response.raise_for_status()
    body = response.json()
    if "error" in body:
        raise RuntimeError(json.dumps(body["error"]))
    return body["result"]


def send_params(text, context_id=None):
    message = {"role": "ROLE_USER", "messageId": str(uuid4()), "parts": [{"text": text}]}
    if context_id:
        message["contextId"] = context_id
    return {"message": message, "configuration": {"historyScope": "task"}}


def task_text(task):
    return "\n".join(part["text"] for artifact in task.get("artifacts", [])
                     for part in artifact.get("parts", []) if "text" in part)

The two graph sources below are the files each server loads. `PARTS_GRAPH` answers from the demo database. `COORDINATOR` finds the specialist's UUID, then posts an A2A 1.0 `SendMessage` request : a JSON-RPC 2.0 envelope with a `ROLE_USER` message, its `messageId`, the coordinator's own thread id as `contextId`, and text parts without `kind`.

In [4]:
PARTS_GRAPH = r"""
import json
import sqlite3
from langchain.messages import SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from server_model import ROOT, llm


class ServedState(MessagesState):
    usage: dict


def answer(state):
    with sqlite3.connect(f"file:{(ROOT / 'data/demo-world/service.db').as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no IN ('CE-CF-02', 'CE-DA-03')").fetchall()
    reply = llm.invoke([SystemMessage("Answer using this inventory only. Retain every count and depot.\n" + json.dumps(rows)),
                        *state["messages"]])
    return {"messages": [reply], "usage": reply.usage_metadata or {}}


builder = StateGraph(ServedState)
builder.add_node("answer", answer)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile()
"""

In [5]:
COORDINATOR = r"""
import json
import os
import time
from uuid import uuid4
import httpx
from langchain.messages import SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from server_model import llm


class CoordinatorState(MessagesState):
    exchange: dict
    usage: dict


def coordinate(state, config):
    context_id = str(config["configurable"]["thread_id"])
    headers = {"Accept": "application/json"}
    with httpx.Client(timeout=180, trust_env=False) as client:
        listed = client.post("http://127.0.0.1:2125/assistants/search", json={"limit": 20})
        listed.raise_for_status()
        assistant_id = next(row["assistant_id"] for row in listed.json() if row.get("graph_id") == "parts")
        payload = {"jsonrpc": "2.0", "id": str(uuid4()), "method": "SendMessage", "params": {
            "message": {"role": "ROLE_USER", "messageId": str(uuid4()), "contextId": context_id,
                        "parts": [{"text": str(state["messages"][-1].content)}]},
            "configuration": {"historyScope": "task"}}}
        response = client.post("http://127.0.0.1:2125/a2a/" + assistant_id, json=payload, headers=headers)
        response.raise_for_status()
    body = response.json()
    if "error" in body:
        raise RuntimeError(json.dumps(body["error"]))
    task = body["result"]["task"]
    if task["status"]["state"] != "TASK_STATE_COMPLETED":
        raise RuntimeError(json.dumps(task["status"]))
    text = "\n".join(part["text"] for artifact in task.get("artifacts", []) for part in artifact["parts"] if "text" in part)
    time.sleep(1 / float(os.environ.get("COURSE_RPS", "2")))
    reply = llm.invoke([SystemMessage("Write a brief handover from the returned evidence only. Preserve counts and depots.\n" + text),
                        *state["messages"]])
    return {"messages": [reply], "exchange": {"request": payload, "response": body, "assistant_id": assistant_id},
            "usage": reply.usage_metadata or {}}


builder = StateGraph(CoordinatorState)
builder.add_node("coordinate", coordinate)
builder.add_edge(START, "coordinate")
builder.add_edge("coordinate", END)
graph = builder.compile()
"""

In [6]:
server_a = NOTEBOOK_DIR / "server-a"
server_b = NOTEBOOK_DIR / "server-b"
for folder in (server_a, server_b):
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "server_model.py").write_text('\nimport runpy\nfrom pathlib import Path\n\nROOT = next(p for p in Path(__file__).resolve().parents if (p / "pyproject.toml").exists())\nsetup = runpy.run_path(str(ROOT / "build" / "setup_cell.py"))\nllm = setup["make_llm"](max_tokens=500, max_retries=0)\n', encoding="utf-8")
(server_b / "parts_graph.py").write_text(PARTS_GRAPH, encoding="utf-8")
(server_a / "coordinator_graph.py").write_text(COORDINATOR, encoding="utf-8")
try:
    parts_url = start_server(2125, {"parts": "./parts_graph.py:graph"}, server_b)
    coordinator_url = start_server(2124, {"coordinator": "./coordinator_graph.py:graph"}, server_a)
    parts_id = assistant_for(parts_url, "parts")
    coordinator_id = assistant_for(coordinator_url, "coordinator")
    for base, assistant in [(parts_url, parts_id), (coordinator_url, coordinator_id)]:
        response = httpx.get(base + "/.well-known/agent-card.json", params={"assistant_id": assistant}, trust_env=False)
        response.raise_for_status()
        print("agent card :", json.dumps(response.json(), indent=2))
    outer = rpc(coordinator_url, coordinator_id, "SendMessage", send_params("Report CE-CF-02 and CE-DA-03 stock and their depots."))["task"]
    print("coordinator state :", outer["status"]["state"])
    print("coordinator answer :", task_text(outer))
    context = outer["contextId"]
    state_response = httpx.get(coordinator_url + "/threads/" + context + "/state", trust_env=False)
    state_response.raise_for_status()
    coordinator_state = state_response.json()["values"]
    print("exchange :", json.dumps(coordinator_state["exchange"], indent=2))
    remote_task = coordinator_state["exchange"]["response"]["result"]["task"]
    print("same context :", context == remote_task["contextId"])
    print("different task :", outer["id"] != remote_task["id"])
    print("coordinator usage :", coordinator_state["usage"])
    remote_state = httpx.get(parts_url + "/threads/" + context + "/state", trust_env=False).json()["values"]
    print("specialist usage :", remote_state["usage"])
    follow = rpc(parts_url, parts_id, "message/send", send_params("Which of those parts is out of stock?", context))
    print("legacy state :", follow["status"]["state"])
    print("legacy keys :", sorted(follow))
    print("follow-up :", task_text(follow))
    print("new task in context :", follow["id"] != remote_task["id"], follow["contextId"] == context)
    whole = rpc(parts_url, parts_id, "GetTask", {"id": follow["id"]})
    scoped = rpc(parts_url, parts_id, "GetTask", {"id": follow["id"], "historyScope": "task"})
    print("history entries :", {"default context": len(whole.get("history", [])), "task": len(scoped.get("history", []))})
except Exception:
    stop_servers()
    raise

server : http://127.0.0.1:2125 ready


server : http://127.0.0.1:2124 ready
graph id : parts
assistant id : b67dba3b-3342-5f55-b1fe-bdcc570cc1a4
uuid5 match : True
graph id : coordinator
assistant id : 1e63a636-5cd8-5425-94a1-d65cc0913fbc
uuid5 match : True
agent card : {
  "name": "parts",
  "description": "parts assistant",
  "url": "http://127.0.0.1:2125/a2a/b67dba3b-3342-5f55-b1fe-bdcc570cc1a4",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2125/a2a/b67dba3b-3342-5f55-b1fe-bdcc570cc1a4",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "required": false,
        "params": {
          "header": "LangGraph-A2A-History-Scope",
          "values": [
            "context",
            "ta

agent card : {
  "name": "coordinator",
  "description": "coordinator assistant",
  "url": "http://127.0.0.1:2124/a2a/1e63a636-5cd8-5425-94a1-d65cc0913fbc",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2124/a2a/1e63a636-5cd8-5425-94a1-d65cc0913fbc",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "required": false,
        "params": {
          "header": "LangGraph-A2A-History-Scope",
          "values": [
            "context",
            "task"
          ],
          "default": "context",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ]
        }
      }
    ]
  },
  "defaultInputModes

coordinator state : TASK_STATE_COMPLETED
coordinator answer : - **CE-CF-02:** 6 units at Rietpoort
- **CE-DA-03:** 0 units at Rietpoort
exchange : {
  "request": {
    "jsonrpc": "2.0",
    "id": "2d14713e-c130-470d-b13f-eeaf084eb837",
    "method": "SendMessage",
    "params": {
      "message": {
        "role": "ROLE_USER",
        "messageId": "89e7fd66-3878-49b6-a5d9-7aa5756859cc",
        "contextId": "af1efca7-ce66-440c-a1e2-23dec8ca26a5",
        "parts": [
          {
            "text": "Report CE-CF-02 and CE-DA-03 stock and their depots."
          }
        ]
      },
      "configuration": {
        "historyScope": "task"
      }
    }
  },
  "response": {
    "jsonrpc": "2.0",
    "id": "2d14713e-c130-470d-b13f-eeaf084eb837",
    "result": {
      "task": {
        "kind": "task",
        "id": "af1efca7-ce66-440c-a1e2-23dec8ca26a5:01a17b68-19d3-7b01-9a23-21c9415c39ea",
        "contextId": "af1efca7-ce66-440c-a1e2-23dec8ca26a5",
        "history": [
          {
        

legacy state : completed
legacy keys : ['artifacts', 'contextId', 'history', 'id', 'kind', 'status']
follow-up : CE-DA-03 is out of stock (0 units) at Rietpoort.
new task in context : True True
history entries : {'default context': 4, 'task': 2}


Each server allocates its own task. Reusing a contextId creates a thread with that id on the second server. The messages stay in that server's storage. A completed task id is not the input of the next turn. The legacy method message/send unwraps the task and writes the state in lowercase. GetTask with the Agent Server option historyScope task keeps the requested task's history. A2A 1.0 itself has historyLength. We request the follow-up task here. This response filter does not limit the model's input history.

We add a request-spacing delay inside the coordinator because the servers have separate process-local model limiters. The delay keeps the combined model traffic within COURSE_RPS. Server failures raise instead of becoming a fabricated stock answer.

## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [7]:
stop_servers()
print("servers stopped :", not servers)
finish_metrics("demo04")

servers stopped : True
metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.4, 'calls': 0, 'input_tokens': 0, 'output_tokens': 0, 'seconds': 23.1}
